In [1]:
# 1. 下載並安裝 ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -r requirements.txt



Cloning into 'ComfyUI'...
remote: Enumerating objects: 35572, done.
remote: Total 35572 (delta 0), reused 0 (delta 0), pack-reused 35572 (from 1)
Receiving objects: 100% (35572/35572), 83.97 MiB | 19.16 MiB/s, done.
Resolving deltas: 100% (24165/24165), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.9/21.9 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 135.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [2]:
# 2. 安裝 ComfyUI-Manager (方便你等一下安裝缺少的節點，例如 VideoHelperSuite)
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
%cd /content/ComfyUI


/content/ComfyUI/custom_nodes
Cloning into 'ComfyUI-Manager'...
remote: Enumerating objects: 29269, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 29269 (delta 26), reused 15 (delta 13), pack-reused 29221 (from 3)
Receiving objects: 100% (29269/29269), 155.65 MiB | 28.04 MiB/s, done.
Resolving deltas: 100% (21708/21708), done.
/content/ComfyUI


In [3]:
# 3. 安裝 Cloudflare Tunnel (用來把 Colab 的網頁變成公開網址讓你連線)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb


Selecting previously unselected package cloudflared.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.3.0) ...
Setting up cloudflared (2026.3.0) ...
Processing triggers for man-db (2.10.2-1) ...


In [10]:
# 1. 安裝更快的下載工具 aria2c (修正為 -y 參數)
!apt-get update && apt-get install -y aria2

# 2. 定義模型下載連結（從 HuggingFace）
unet_model = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_i2v_720p_14B_fp8_scaled.safetensors?download=true"
clip_model = "https://huggingface.co/chatpig/encoder/resolve/main/umt5_xxl_fp8_e4m3fn_scaled.safetensors?download=true"
clip_vision_model = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors?download=true"
vae_model = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors?download=true"

# 3. 確保 ComfyUI 資料夾結構正確（已修改為你工作流指定的正確路徑）
%cd /content/ComfyUI
!mkdir -p models/diffusion_models
!mkdir -p models/text_encoders
!mkdir -p models/clip_vision
!mkdir -p models/vae

# 4. 開始自動下載（使用 16 線程加速）
print("🚀 開始下載 UNet 模型...")
!aria2c -x 16 -s 16 -d /content/ComfyUI/models/diffusion_models -o wan2.1_i2v_720p_14B_fp8_e4m3fn.safetensors $unet_model

print("🚀 開始下載 CLIP 模型...")
!aria2c -x 16 -s 16 -d /content/ComfyUI/models/text_encoders -o umt5_xxl_fp8_e4m3fn_scaled.safetensors $clip_model

print("🚀 開始下載 CLIP Vision 模型...")
!aria2c -x 16 -s 16 -d /content/ComfyUI/models/clip_vision -o clip_vision_h.safetensors $clip_vision_model

print("🚀 開始下載 VAE 模型...")
!aria2c -x 16 -s 16 -d /content/ComfyUI/models/vae -o wan_2.1_vae.safetensors $vae_model

print("\n" + "="*50)
print("✅ 所有 4 個核心模型下載完成！且已放入正確的資料夾！")
print("👉 接下來請記得去「重啟 ComfyUI 伺服器」讓系統重新掃描模型。")
print("="*50 + "\n")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [12]:
# 切換到自定義節點資料夾
%cd /content/ComfyUI/custom_nodes

# 下載 VideoHelperSuite 插件
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git

# 進入插件資料夾並安裝它需要的依賴套件
%cd ComfyUI-VideoHelperSuite
!pip install -r requirements.txt

print("\n" + "="*50)
print("✅ VideoHelperSuite 節點安裝完成！")
print("👉 請去重啟 ComfyUI 伺服器，然後重新整理網頁。")
print("="*50 + "\n")

/content/ComfyUI/custom_nodes
Cloning into 'ComfyUI-VideoHelperSuite'...
remote: Enumerating objects: 3345, done.
remote: Counting objects: 100% (1594/1594), done.
remote: Compressing objects: 100% (369/369), done.
remote: Total 3345 (delta 1456), reused 1228 (delta 1223), pack-reused 1751 (from 3)
Receiving objects: 100% (3345/3345), 793.63 KiB | 17.64 MiB/s, done.
Resolving deltas: 100% (1972/1972), done.
/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite

✅ VideoHelperSuite 節點安裝完成！
👉 請去重啟 ComfyUI 伺服器，然後重新整理網頁。



In [ ]:
# 4. 啟動 ComfyUI 並產生對外網址
# 安裝 localtunnel
%cd /content/ComfyUI
!npm install -g localtunnel
import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()

    print("\n" + "="*50)
    print("✅ ComfyUI 準備就緒！")
    print("⚠️ 進入網址後，系統會要求輸入密碼 (Endpoint IP)，請複製下面這串數字：")
    req = urllib.request.Request('https://ipv4.icanhazip.com')
    print("👉👉👉", urllib.request.urlopen(req).read().decode('utf8').strip(), "👈👈👈")
    print("="*50 + "\n")

    p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
    for line in p.stdout:
        print("🔗 你的專屬網址 =>", line.decode(), end='')

# 啟動背景網址服務
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

# 啟動 ComfyUI
!python main.py --dont-print-server

/content/ComfyUI
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
changed 22 packages in 2s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-03-29 08:06:43.591
** Platform: Linux
** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log

Prestartup times for custom nodes:
   6.2 seconds: /content/ComfyUI/custom_nodes/ComfyUI-Manager

Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8']